# SPR 2026 - Mammography Report Classification
## Google Colab Experiments

**Metrica**: Macro F1 | **Baseline publico**: 0.773 (TF-IDF + LinearSVC)

---

### Experimentos

| Secao | Experimento | Modelo | GPU RAM |
|-------|-------------|--------|---------|
| EXP-00 | Baseline TF-IDF | LinearSVC + LR | CPU |
| EXP-01 | BERTimbau Base | neuralmind/bert-base-portuguese-cased | ~6 GB |
| EXP-03 | BioBERTpt | pucpr/biobertpt-all | ~6 GB |
| EXP-04 | mDeBERTa-v3 | microsoft/mdeberta-v3-base | ~8 GB |
| EXP-05 | Ensemble BERT+TF-IDF | melhor BERT + TF-IDF | ~6 GB |

> **Dica**: Execute uma secao por vez. Cada EXP e independente.

---
## SETUP - Rodar sempre primeiro

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'GPU nao encontrada')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece protobuf
!pip install -q kaggle
print('Dependencias instaladas')

In [ ]:
from google.colab import files
import os

os.makedirs('/root/.kaggle', exist_ok=True)
print('Faca upload do seu kaggle.json:')
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json configurado')
else:
    print('ERRO: kaggle.json nao encontrado')

In [ ]:
import os
os.makedirs('data', exist_ok=True)
!kaggle competitions download -c spr-2026-mammography-report-classification -p data/
!unzip -o data/spr-2026-mammography-report-classification.zip -d data/

import pandas as pd
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
print(f'train: {train.shape}, test: {test.shape}')
print(f'Colunas: {list(train.columns)}')
print('Distribuicao de classes:')
print(train['target'].value_counts().sort_index())

In [ ]:
import hashlib, csv, os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

RANDOM_STATE = 42
N_SPLITS = 5
RESULTS_PATH = 'results.tsv'

def stable_hash(s):
    return hashlib.md5(s.encode('utf-8')).hexdigest()

def make_groups(df):
    return df['report'].apply(stable_hash).values

def evaluate(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

def save_submission(test_df, preds, path='submission.csv'):
    sub = pd.DataFrame({'ID': test_df['ID'], 'target': preds.astype(int)})
    sub.to_csv(path, index=False)
    print(f'Submission salvo: {path} ({len(sub)} linhas)')
    return sub

def log_result(exp_id, model, cv_f1, vram_gb, time_min, status, notes):
    file_exists = os.path.exists(RESULTS_PATH)
    with open(RESULTS_PATH, 'a', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        if not file_exists:
            w.writerow(['exp_id','model','cv_f1','vram_gb','time_min','status','notes'])
        w.writerow([exp_id, model, f'{cv_f1:.5f}', vram_gb, time_min, status, notes])
    print(f'Logado: {exp_id} | cv_f1={cv_f1:.5f} | {status}')

print('Funcoes auxiliares carregadas')

---
## EXP-00 - Baseline TF-IDF + Ensemble (CPU)

> Replica o train.py atual. Roda em CPU, ~2 min.
> **Meta**: confirmar cv_f1 aprox 0.748 no Colab.

In [ ]:
import re, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
warnings.filterwarnings('ignore')

_ABBREV = {
    r'\bbi-?rads\b': 'birads',
    r'\bcalc\.?\b': 'calcificacao',
    r'\bnod\.?\b': 'nodulo',
    r'\bdx\.?\b': 'diagnostico',
    r'\blt\.?\b': 'lateral',
    r'\bcc\b': 'craniocaudal',
    r'\bmlo\b': 'mediolateral',
}

def clean_text(s):
    if pd.isna(s): return ''
    s = str(s).strip().lower()
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n{2,}', '\n', s)
    for pat, rep in _ABBREV.items():
        s = re.sub(pat, rep, s)
    return s

def build_vec():
    return FeatureUnion([
        ('word', TfidfVectorizer(
            ngram_range=(1,3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ('char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3,5), min_df=3,
            max_df=0.95, sublinear_tf=True, max_features=80000)),
    ])

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
train_df['text'] = train_df['report'].apply(clean_text)
test_df['text']  = test_df['report'].apply(clean_text)

X_text  = train_df['text'].values
y       = train_df['target'].astype(int).values
groups  = make_groups(train_df)
classes = sorted(np.unique(y))
gkf     = GroupKFold(n_splits=N_SPLITS)
oof_svc = np.zeros((len(train_df), len(classes)))
oof_lr  = np.zeros((len(train_df), len(classes)))

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_text, y, groups)):
    vec  = build_vec()
    X_tr = vec.fit_transform(X_text[tr_idx])
    X_v  = vec.transform(X_text[val_idx])

    svc = LinearSVC(C=1.0, class_weight='balanced',
                    random_state=RANDOM_STATE, max_iter=10000)
    svc.fit(X_tr, y[tr_idx])
    dec = svc.decision_function(X_v)
    dec_exp = np.exp(dec - dec.max(axis=1, keepdims=True))
    oof_svc[val_idx] = dec_exp / dec_exp.sum(axis=1, keepdims=True)

    lr = LogisticRegression(C=4.0, class_weight='balanced', solver='saga',
                            max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
    lr.fit(X_tr, y[tr_idx])
    oof_lr[val_idx] = lr.predict_proba(X_v)

    ens   = 0.5*oof_svc[val_idx] + 0.5*oof_lr[val_idx]
    preds = np.array(classes)[np.argmax(ens, axis=1)]
    print(f'Fold {fold+1}: macro F1 = {evaluate(y[val_idx], preds):.5f}')

oof_final = 0.5*oof_svc + 0.5*oof_lr
oof_preds = np.array(classes)[np.argmax(oof_final, axis=1)]
cv_score  = evaluate(y, oof_preds)
elapsed   = (time.time() - t0) / 60
print(f'CV macro F1: {cv_score:.5f} | Tempo: {elapsed:.1f} min')
log_result('EXP-00','TF-IDF+SVC+LR',cv_score,0,round(elapsed,1),'keep','baseline colab')

---
## EXP-01 - BERTimbau Base Fine-Tuning

**Modelo**: `neuralmind/bert-base-portuguese-cased`  
**VRAM**: ~6 GB | **Tempo**: ~40-60 min no T4

> Requer GPU: Runtime > Change runtime type > T4 GPU

In [ ]:
EXP_ID     = 'EXP-01'
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
NUM_LABELS = 7
MAX_LEN    = 256
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 2e-5
WARMUP     = 0.1
WEIGHT_DECAY = 0.01
print(f'Config: {MODEL_NAME} | max_len={MAX_LEN} | batch={BATCH_SIZE} | epochs={EPOCHS} | lr={LR}')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReportDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = labels
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f'Tokenizer carregado: {MODEL_NAME}')

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
groups   = make_groups(train_df)
classes  = sorted(np.unique(y))

gkf       = GroupKFold(n_splits=N_SPLITS)
oof_proba = np.zeros((len(train_df), NUM_LABELS))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'macro_f1': evaluate(labels, np.argmax(logits, axis=1))}

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_text, y, groups)):
    print(f'Fold {fold+1}/{N_SPLITS}')
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    total_steps  = (len(tr_idx) // BATCH_SIZE) * EPOCHS
    warmup_steps = int(total_steps * WARMUP)
    ga = getattr(globals(), 'GRAD_ACCUM', 1)
    args = TrainingArguments(
        output_dir=f'checkpoints/{EXP_ID}_fold{fold+1}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=ga,
        learning_rate=LR, weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        evaluation_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        fp16=True, dataloader_num_workers=2,
        report_to='none', logging_steps=50,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=ReportDataset(X_text[tr_idx], y[tr_idx]),
        eval_dataset=ReportDataset(X_text[val_idx], y[val_idx]),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    pred_out = trainer.predict(ReportDataset(X_text[val_idx], y[val_idx]))
    proba = torch.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()
    oof_proba[val_idx] = proba
    fp = np.array(classes)[np.argmax(proba, axis=1)]
    print(f'  macro F1: {evaluate(y[val_idx], fp):.5f}')
    del model, trainer
    torch.cuda.empty_cache()

oof_preds = np.array(classes)[np.argmax(oof_proba, axis=1)]
cv_score  = evaluate(y, oof_preds)
elapsed   = (time.time() - t0) / 60
vram_used = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'CV macro F1: {cv_score:.5f} | Tempo: {elapsed:.1f} min | VRAM: {vram_used:.1f} GB')
log_result(EXP_ID, MODEL_NAME.split('/')[-1], cv_score,
           round(vram_used,1), round(elapsed,1),
           'keep' if cv_score > 0.748 else 'review',
           f'lr={LR} ep={EPOCHS} max_len={MAX_LEN}')
np.save(f'oof_proba_{EXP_ID}.npy', oof_proba)
print(f'OOF salvo: oof_proba_{EXP_ID}.npy')

In [ ]:
# Gerar submission no dataset completo
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
classes  = sorted(np.unique(y))

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
)
args_final = TrainingArguments(
    output_dir=f'checkpoints/{EXP_ID}_final',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR, weight_decay=WEIGHT_DECAY,
    fp16=True, report_to='none', logging_steps=100, save_strategy='no',
)
trainer_final = Trainer(
    model=model_final, args=args_final,
    train_dataset=ReportDataset(X_text, y)
)
trainer_final.train()
pred_out   = trainer_final.predict(ReportDataset(test_df['report'].fillna('').values))
test_preds = np.array(classes)[np.argmax(pred_out.predictions, axis=1)]
save_submission(test_df, test_preds, path=f'submission_{EXP_ID}.csv')

---
## EXP-03 - BioBERTpt Fine-Tuning

**Modelo**: `pucpr/biobertpt-all`  
BERT biomedico em portugues - treinado em PubMed + SciELO PT-BR  
**VRAM**: ~6 GB | **Tempo**: ~40-60 min no T4

In [ ]:
EXP_ID     = 'EXP-03'
MODEL_NAME = 'pucpr/biobertpt-all'
NUM_LABELS = 7
MAX_LEN    = 256
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 2e-5
WARMUP     = 0.1
WEIGHT_DECAY = 0.01
print(f'Config: {MODEL_NAME} | max_len={MAX_LEN} | batch={BATCH_SIZE} | epochs={EPOCHS} | lr={LR}')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReportDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = labels
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f'Tokenizer carregado: {MODEL_NAME}')

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
groups   = make_groups(train_df)
classes  = sorted(np.unique(y))

gkf       = GroupKFold(n_splits=N_SPLITS)
oof_proba = np.zeros((len(train_df), NUM_LABELS))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'macro_f1': evaluate(labels, np.argmax(logits, axis=1))}

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_text, y, groups)):
    print(f'Fold {fold+1}/{N_SPLITS}')
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    total_steps  = (len(tr_idx) // BATCH_SIZE) * EPOCHS
    warmup_steps = int(total_steps * WARMUP)
    ga = getattr(globals(), 'GRAD_ACCUM', 1)
    args = TrainingArguments(
        output_dir=f'checkpoints/{EXP_ID}_fold{fold+1}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=ga,
        learning_rate=LR, weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        evaluation_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        fp16=True, dataloader_num_workers=2,
        report_to='none', logging_steps=50,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=ReportDataset(X_text[tr_idx], y[tr_idx]),
        eval_dataset=ReportDataset(X_text[val_idx], y[val_idx]),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    pred_out = trainer.predict(ReportDataset(X_text[val_idx], y[val_idx]))
    proba = torch.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()
    oof_proba[val_idx] = proba
    fp = np.array(classes)[np.argmax(proba, axis=1)]
    print(f'  macro F1: {evaluate(y[val_idx], fp):.5f}')
    del model, trainer
    torch.cuda.empty_cache()

oof_preds = np.array(classes)[np.argmax(oof_proba, axis=1)]
cv_score  = evaluate(y, oof_preds)
elapsed   = (time.time() - t0) / 60
vram_used = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'CV macro F1: {cv_score:.5f} | Tempo: {elapsed:.1f} min | VRAM: {vram_used:.1f} GB')
log_result(EXP_ID, MODEL_NAME.split('/')[-1], cv_score,
           round(vram_used,1), round(elapsed,1),
           'keep' if cv_score > 0.748 else 'review',
           f'lr={LR} ep={EPOCHS} max_len={MAX_LEN}')
np.save(f'oof_proba_{EXP_ID}.npy', oof_proba)
print(f'OOF salvo: oof_proba_{EXP_ID}.npy')

In [ ]:
# Gerar submission no dataset completo
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
classes  = sorted(np.unique(y))

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
)
args_final = TrainingArguments(
    output_dir=f'checkpoints/{EXP_ID}_final',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR, weight_decay=WEIGHT_DECAY,
    fp16=True, report_to='none', logging_steps=100, save_strategy='no',
)
trainer_final = Trainer(
    model=model_final, args=args_final,
    train_dataset=ReportDataset(X_text, y)
)
trainer_final.train()
pred_out   = trainer_final.predict(ReportDataset(test_df['report'].fillna('').values))
test_preds = np.array(classes)[np.argmax(pred_out.predictions, axis=1)]
save_submission(test_df, test_preds, path=f'submission_{EXP_ID}.csv')

---
## EXP-04 - mDeBERTa-v3 Fine-Tuning

**Modelo**: `microsoft/mdeberta-v3-base`  
**VRAM**: ~8 GB | **Tempo**: ~50-70 min no T4

> BATCH_SIZE=16 + GRAD_ACCUM=2 para caber no T4 (equivale a batch=32)

In [ ]:
EXP_ID     = 'EXP-04'
MODEL_NAME = 'microsoft/mdeberta-v3-base'
NUM_LABELS = 7
MAX_LEN    = 512
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS     = 5
LR         = 1e-5
WARMUP     = 0.1
WEIGHT_DECAY = 0.01
print(f'Config: {MODEL_NAME} | max_len={MAX_LEN} | batch={BATCH_SIZE}x{GRAD_ACCUM} | lr={LR}')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReportDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = labels
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f'Tokenizer carregado: {MODEL_NAME}')

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
groups   = make_groups(train_df)
classes  = sorted(np.unique(y))

gkf       = GroupKFold(n_splits=N_SPLITS)
oof_proba = np.zeros((len(train_df), NUM_LABELS))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'macro_f1': evaluate(labels, np.argmax(logits, axis=1))}

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_text, y, groups)):
    print(f'Fold {fold+1}/{N_SPLITS}')
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    total_steps  = (len(tr_idx) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    warmup_steps = int(total_steps * WARMUP)
    args = TrainingArguments(
        output_dir=f'checkpoints/{EXP_ID}_fold{fold+1}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        evaluation_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        fp16=True, dataloader_num_workers=2,
        report_to='none', logging_steps=50,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=ReportDataset(X_text[tr_idx], y[tr_idx]),
        eval_dataset=ReportDataset(X_text[val_idx], y[val_idx]),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    pred_out = trainer.predict(ReportDataset(X_text[val_idx], y[val_idx]))
    proba = torch.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()
    oof_proba[val_idx] = proba
    fp = np.array(classes)[np.argmax(proba, axis=1)]
    print(f'  macro F1: {evaluate(y[val_idx], fp):.5f}')
    del model, trainer
    torch.cuda.empty_cache()

oof_preds = np.array(classes)[np.argmax(oof_proba, axis=1)]
cv_score  = evaluate(y, oof_preds)
elapsed   = (time.time() - t0) / 60
vram_used = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'CV macro F1: {cv_score:.5f} | Tempo: {elapsed:.1f} min | VRAM: {vram_used:.1f} GB')
log_result(EXP_ID, 'mdeberta-v3-base', cv_score,
           round(vram_used,1), round(elapsed,1),
           'keep' if cv_score > 0.748 else 'review',
           f'lr={LR} ep={EPOCHS} max_len={MAX_LEN} ga={GRAD_ACCUM}')
np.save(f'oof_proba_{EXP_ID}.npy', oof_proba)
print(f'OOF salvo: oof_proba_{EXP_ID}.npy')

In [ ]:
# Gerar submission no dataset completo
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')
X_text   = train_df['report'].fillna('').values
y        = train_df['target'].astype(int).values
classes  = sorted(np.unique(y))

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
)
args_final = TrainingArguments(
    output_dir=f'checkpoints/{EXP_ID}_final',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR, weight_decay=WEIGHT_DECAY,
    fp16=True, report_to='none', logging_steps=100, save_strategy='no',
)
trainer_final = Trainer(
    model=model_final, args=args_final,
    train_dataset=ReportDataset(X_text, y)
)
trainer_final.train()
pred_out   = trainer_final.predict(ReportDataset(test_df['report'].fillna('').values))
test_preds = np.array(classes)[np.argmax(pred_out.predictions, axis=1)]
save_submission(test_df, test_preds, path=f'submission_{EXP_ID}.csv')

---
## EXP-05 - Ensemble BERT + TF-IDF

Combina OOF do melhor BERT com TF-IDF baseline.  
BERT captura semantica; TF-IDF captura padroes lexicos raros (classes 5 e 6).

> Pre-requisito: Ter rodado EXP-00 + pelo menos um EXP-01/03/04

In [ ]:
BERT_EXP    = 'EXP-01'  # ou 'EXP-03' ou 'EXP-04'
ALPHA_BERT  = 0.7
ALPHA_TFIDF = 0.3

import os
oof_file = f'oof_proba_{BERT_EXP}.npy'
if os.path.exists(oof_file):
    print(f'Usando OOF de {BERT_EXP}: {oof_file}')
else:
    print(f'ERRO: {oof_file} nao encontrado - rode o {BERT_EXP} primeiro')

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
warnings.filterwarnings('ignore')

def clean_text(s):
    if pd.isna(s): return ''
    s = str(s).strip().lower()
    for pat, rep in {
        r'\bbi-?rads\b': 'birads',
        r'\bcalc\.?\b': 'calcificacao',
        r'\bnod\.?\b': 'nodulo',
        r'\bdx\.?\b': 'diagnostico',
    }.items():
        s = re.sub(pat, rep, s)
    return s

train_df = pd.read_csv('data/train.csv')
train_df['text'] = train_df['report'].apply(clean_text)
X_text  = train_df['text'].values
y       = train_df['target'].astype(int).values
groups  = make_groups(train_df)
classes = sorted(np.unique(y))

gkf     = GroupKFold(n_splits=N_SPLITS)
oof_svc = np.zeros((len(train_df), len(classes)))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_text, y, groups)):
    vec = FeatureUnion([
        ('word', TfidfVectorizer(
            ngram_range=(1,3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ('char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3,5), min_df=3,
            max_df=0.95, sublinear_tf=True, max_features=80000)),
    ])
    X_tr = vec.fit_transform(X_text[tr_idx])
    X_v  = vec.transform(X_text[val_idx])
    svc  = LinearSVC(C=1.0, class_weight='balanced', random_state=42, max_iter=10000)
    svc.fit(X_tr, y[tr_idx])
    dec = svc.decision_function(X_v)
    dec_exp = np.exp(dec - dec.max(axis=1, keepdims=True))
    oof_svc[val_idx] = dec_exp / dec_exp.sum(axis=1, keepdims=True)

oof_bert  = np.load(oof_file)
oof_final = ALPHA_BERT * oof_bert + ALPHA_TFIDF * oof_svc
oof_preds = np.array(classes)[np.argmax(oof_final, axis=1)]
cv_score  = evaluate(y, oof_preds)

f1_bert  = evaluate(y, np.array(classes)[np.argmax(oof_bert, axis=1)])
f1_tfidf = evaluate(y, np.array(classes)[np.argmax(oof_svc, axis=1)])

print(f'TF-IDF macro F1: {f1_tfidf:.5f}')
print(f'BERT   macro F1: {f1_bert:.5f}')
print(f'Ensemble macro F1: {cv_score:.5f}  (alpha_bert={ALPHA_BERT})')

log_result('EXP-05', f'ensemble-{BERT_EXP}+tfidf', cv_score, 0, 0,
           'keep' if cv_score > max(f1_bert, f1_tfidf) else 'review',
           f'alpha_bert={ALPHA_BERT} alpha_tfidf={ALPHA_TFIDF}')

---
## Resultados e Persistencia

In [ ]:
import pandas as pd, os
if os.path.exists('results.tsv'):
    df = pd.read_csv('results.tsv', sep='\t')
    print(df.sort_values('cv_f1', ascending=False).to_string(index=False))
else:
    print('Nenhum resultado ainda')

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/SPR2026'
os.makedirs(DRIVE_PATH, exist_ok=True)

to_save = ['results.tsv'] + [
    f for f in os.listdir('.')
    if f.startswith(('submission_', 'oof_proba_'))
]
for fname in to_save:
    if os.path.exists(fname):
        shutil.copy(fname, f'{DRIVE_PATH}/{fname}')
        print(f'Salvo no Drive: {fname}')

print(os.listdir(DRIVE_PATH))

In [ ]:
EXP_TO_SUBMIT = 'EXP-01'  # editar para o melhor experimento
sub_file = f'submission_{EXP_TO_SUBMIT}.csv'
import os
if os.path.exists(sub_file):
    get_ipython().system(
        'kaggle competitions submit'
        ' -c spr-2026-mammography-report-classification'
        f' -f {sub_file}'
        f' -m "Colab {EXP_TO_SUBMIT}"'
    )
    print(f'Submission enviado: {sub_file}')
else:
    print(f'ERRO: {sub_file} nao encontrado')